In [37]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [38]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls = PyPDFLoader
    )

    documents = loader.load()
    return documents

In [39]:
extracted_docs =load_pdf_files("../data")

In [40]:
extracted_docs

[Document(metadata={'producer': 'Adobe PDF Library 16.0.3', 'creator': 'Adobe InDesign 17.0 (Windows)', 'creationdate': '2023-12-14T16:44:52+05:30', 'moddate': '2024-01-22T08:55:22+05:30', 'trapped': '/False', 'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf', 'total_pages': 72, 'page': 0, 'page_label': '1'}, page_content='National Health Promotion Programme\nSri Lanka\nStrategic Plan 2024 -2030\n  \nHealth Promotion Bureau\nMinistry of Health\nSri Lanka\n2024\nISBN 978-624-5719-97-6\nPrinted by :\nTharanjee Prints (Pvt) Ltd\n506, Highlevel Road,\nNawinna, Maharagama.'),
 Document(metadata={'producer': 'Adobe PDF Library 16.0.3', 'creator': 'Adobe InDesign 17.0 (Windows)', 'creationdate': '2023-12-14T16:44:52+05:30', 'moddate': '2024-01-22T08:55:22+05:30', 'trapped': '/False', 'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf', 'total_pages': 72, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Librar

In [41]:
len(extracted_docs)

72

In [42]:
#filtering metadata from the docs
from langchain.schema import Document

def filter_metadata(docs):
    minimal_docs = []

    for doc in docs:
        source = doc.metadata.get("source")

        new_doc = Document(
            page_content=doc.page_content,
            metadata={"source": source}
        )

        minimal_docs.append(new_doc)

    return minimal_docs


In [43]:
minimal_docs = filter_metadata(extracted_docs)

In [44]:
minimal_docs

[Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='National Health Promotion Programme\nSri Lanka\nStrategic Plan 2024 -2030\n  \nHealth Promotion Bureau\nMinistry of Health\nSri Lanka\n2024\nISBN 978-624-5719-97-6\nPrinted by :\nTharanjee Prints (Pvt) Ltd\n506, Highlevel Road,\nNawinna, Maharagama.'),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='- 1 -\nNational Health Promotion Programme\nSri Lanka\nStrategic Plan 2024 -2030  \nHealth Promotion Bureau\nMinistry of Health\n2024'),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='- 2 -'),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='- 3 -\nTable of contents\nList of Abbreviations ... ... .

In [45]:
#split the doc into small chunks

def text_splitter(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20,
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [46]:
text_chunks = text_splitter(minimal_docs)
print(f"Number of chunks: {len(text_chunks)}")

Number of chunks: 296


In [47]:
text_chunks

[Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='National Health Promotion Programme\nSri Lanka\nStrategic Plan 2024 -2030\n  \nHealth Promotion Bureau\nMinistry of Health\nSri Lanka\n2024\nISBN 978-624-5719-97-6\nPrinted by :\nTharanjee Prints (Pvt) Ltd\n506, Highlevel Road,\nNawinna, Maharagama.'),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='- 1 -\nNational Health Promotion Programme\nSri Lanka\nStrategic Plan 2024 -2030  \nHealth Promotion Bureau\nMinistry of Health\n2024'),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='- 2 -'),
 Document(metadata={'source': '..\\data\\National-Health-Promotion-Programme-Sri-Lanka.pdf'}, page_content='- 3 -\nTable of contents\nList of Abbreviations ... ... ... ... 4\nForeword  ... ... ... ... 5\nNational Health Promotion Programme ... ... ... 7\nNational Health Promot

In [48]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceBgeEmbeddings(
        model_name = model_name
    )
    return embeddings

embedding = download_embeddings()

In [49]:
embedding

HuggingFaceBgeEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_instruction='Represent this question for searching relevant passages: ', embed_instruction='', show_progress=False)

In [50]:
vector = embedding.embed_query("hello")
vector

[-0.022026540711522102,
 0.19752855598926544,
 0.043609246611595154,
 0.02690303511917591,
 -0.05324876680970192,
 0.02599542774260044,
 0.051608629524707794,
 -0.0028309684712439775,
 0.007863445207476616,
 -0.01641920953989029,
 0.031907010823488235,
 -0.05958908423781395,
 -0.010614817030727863,
 -0.031593337655067444,
 0.08730670809745789,
 -0.02161256968975067,
 -0.047145310789346695,
 -0.00732298893854022,
 -0.06156802549958229,
 -0.02502180077135563,
 0.05932354927062988,
 0.08992072939872742,
 -0.019541621208190918,
 0.0014421336818486452,
 -0.0560927577316761,
 -0.017460571601986885,
 0.024333719164133072,
 -0.03381690755486488,
 0.13417772948741913,
 -0.0450061671435833,
 -0.03773840516805649,
 -0.004884730558842421,
 0.09256843477487564,
 0.03456860035657883,
 -0.0035387706011533737,
 0.08763079345226288,
 0.04665107652544975,
 0.004822831600904465,
 0.05373053997755051,
 -0.08265836536884308,
 -0.022768385708332062,
 -0.04483291134238243,
 -0.013418452814221382,
 0.01169497

In [51]:
vector_dimension = len(vector)
vector_dimension

384

In [52]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [53]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY


In [54]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)


In [55]:
pc

In [56]:
from pinecone import ServerlessSpec
index_name = "medical-public-health-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension= vector_dimension, #dimension of the embeddings
        metric= "cosine", #cosine similarity
        spec = ServerlessSpec(cloud="aws",region="us-east-1")
    )

index = pc.Index(index_name)

In [57]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embedding,
    index_name=index_name
)


In [ ]:
#Loading existing index
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)